In [1]:
import warnings
from rdkit import RDLogger

# 屏蔽 RDKit 警告
RDLogger.DisableLog('rdApp.*')

# 或屏蔽所有 Python 警告
warnings.filterwarnings("ignore")
# 屏蔽 LightGBM 警告
warnings.filterwarnings("ignore", category=UserWarning, module="lightgbm")

In [2]:
import torch
from sklearn.model_selection import StratifiedKFold
import pandas as pd
import numpy as np
from rdkit import Chem
from rdkit.Chem import AllChem
from sklearn.metrics import precision_recall_curve, auc
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
import joblib
import optuna
from rdkit.Chem import Descriptors, AllChem
from tqdm import tqdm  # 导入tqdm
from sklearn.preprocessing import StandardScaler, MinMaxScaler, OneHotEncoder
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import GroupKFold





In [3]:
# 函数：将SMILES转换为分子描述符和指纹
def smiles_to_features(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    # 提取描述符
    descriptors = [
        Descriptors.MolWt(mol),  # 分子量
        Descriptors.MolLogP(mol),  # LogP
        Descriptors.NumHDonors(mol),  # 氢键供体数量
        Descriptors.NumHAcceptors(mol)  # 氢键受体数量
    ]
    # 生成Morgan指纹
    fingerprint = AllChem.GetMorganFingerprintAsBitVect(mol, 2, nBits=2048)
    fingerprint_array = np.zeros((2048,))
    Chem.DataStructs.ConvertToNumpyArray(fingerprint, fingerprint_array)
    # 合并描述符和指纹
    features = np.concatenate([descriptors, fingerprint_array])
    return features


In [4]:
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import GroupKFold
from tqdm import tqdm
import optuna
import numpy as np

def train_evaluate_regression_model_with_optuna(model_name, model_class, param_func, X, y, groups):
    def objective(trial):
        params = param_func(trial)
        model = model_class(**params)

        gkf = GroupKFold(n_splits=10)
        maes = []

        for train_idx, val_idx in tqdm(gkf.split(X, y, groups=groups), total=10, desc=f"Training {model_name}"):
            X_train, X_val = X[train_idx], X[val_idx]
            y_train, y_val = y[train_idx], y[val_idx]

            model.fit(X_train, y_train)
            y_pred = model.predict(X_val)

            # ✅ 计算 MAE
            mae = mean_absolute_error(y_val, y_pred)
            maes.append(mae)

        return np.mean(maes)

    study = optuna.create_study(direction='minimize')
    study.optimize(objective, n_trials=30)

    print(f'Best parameters for {model_name}: {study.best_params}')
    print(f'Best mean MAE: {study.best_value:.4f}')

In [5]:
# 数据预处理
df = pd.read_excel('../fish_EC10_unique.xlsx')
labels = df['mgperL'].values
smiles_list = df['SMILES_Canonical_RDKit'].tolist()
endpoints_a = df['endpoint']
Duration_Values_a = df['Duration_Value'].values
effects_a = df['effect']


In [6]:

features = []
new_labels = []
new_smiles_list = []
endpoints = []
Duration_Values = []
effects =[]


for smiles, label,a,b,c in zip(smiles_list, labels,Duration_Values_a,effects_a,endpoints_a):
    feature = smiles_to_features(smiles)
    if feature is not None:
        features.append(feature)
        new_labels.append(label)
        new_smiles_list.append(smiles)
        Duration_Values.append(a)
        effects.append(b)
        endpoints.append(c)

X = np.array(features)
y = np.array(new_labels)
groups = new_smiles_list  # 可直接用于 GroupKFold




In [7]:
def encode_column(zz):
    zz_series = pd.Series(zz)  # 转换为 Series
    unique_values = zz_series.unique()
    if len(unique_values) > 1:
        encoder = OneHotEncoder(sparse_output=False)
        return encoder.fit_transform(zz_series.values.reshape(-1, 1))
    else:
        return None  # 只有一种类别时忽略

Duration_Values =pd.Series(Duration_Values)


# 编码 effect、endpoint 和 species_group 列
effect_encoded = encode_column(effects)
endpoint_encoded = encode_column(endpoints)
#species_encoded = encode_column(df, 'species_group')

# # 将需要的列拼接成输入 X
X = np.hstack((X, Duration_Values.values.reshape(-1, 1)))

# # 拼接编码后的列（如果存在）
for encoded_feature in [effect_encoded, endpoint_encoded]:
     if encoded_feature is not None:
         X = np.hstack((X, encoded_feature))



y=np.log1p(y)

In [11]:
def xgb_param_func(trial):
    return {
        'n_estimators': trial.suggest_int('n_estimators', 100, 600),
        'max_depth': trial.suggest_int('max_depth', 5, 20),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'reg_alpha': trial.suggest_float('reg_alpha', 0.0, 1.0),   # L1 正则
        'reg_lambda': trial.suggest_float('reg_lambda', 0.0, 1.0)  # L2 正则
    }
from xgboost import XGBRegressor

train_evaluate_regression_model_with_optuna(
    "XGBoost",
    XGBRegressor,
    xgb_param_func,
    X, y, groups
)

[I 2025-05-11 18:56:51,855] A new study created in memory with name: no-name-6b80bec6-7ae1-4e9e-8a26-c5364bbfc099
Training XGBoost: 100%|██████████| 10/10 [00:39<00:00,  3.94s/it]
[I 2025-05-11 18:57:31,254] Trial 0 finished with value: 0.9110637648960672 and parameters: {'n_estimators': 182, 'max_depth': 11, 'learning_rate': 0.18536700453255883, 'subsample': 0.7753207004269773, 'colsample_bytree': 0.9181661701700949, 'reg_alpha': 0.7661347221037444, 'reg_lambda': 0.6818991015910731}. Best is trial 0 with value: 0.9110637648960672.
Training XGBoost: 100%|██████████| 10/10 [01:12<00:00,  7.24s/it]
[I 2025-05-11 18:58:43,652] Trial 1 finished with value: 0.8901379091099928 and parameters: {'n_estimators': 474, 'max_depth': 9, 'learning_rate': 0.10026862956816927, 'subsample': 0.7698836684347512, 'colsample_bytree': 0.9839959430029962, 'reg_alpha': 0.3992256838224987, 'reg_lambda': 0.9066468412101159}. Best is trial 1 with value: 0.8901379091099928.
Training XGBoost: 100%|██████████| 10/1

Best parameters for XGBoost: {'n_estimators': 344, 'max_depth': 19, 'learning_rate': 0.03291635921718668, 'subsample': 0.663989226699964, 'colsample_bytree': 0.7190676735044167, 'reg_alpha': 0.03689122165820412, 'reg_lambda': 0.9983905475773859}
Best mean MAE: 0.8703


In [12]:
from lightgbm import LGBMRegressor

def lgbm_param_func(trial):
    return {
        'n_estimators': trial.suggest_int('n_estimators', 100, 600),
        'max_depth': trial.suggest_int('max_depth', 5, 20),
        'num_leaves': trial.suggest_int('num_leaves', 20, 300),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'feature_fraction': trial.suggest_float('feature_fraction', 0.6, 1.0),
        'bagging_fraction': trial.suggest_float('bagging_fraction', 0.6, 1.0),
        'bagging_freq': trial.suggest_int('bagging_freq', 1, 7),
        'reg_alpha': trial.suggest_float('reg_alpha', 0.0, 1.0),
        'reg_lambda': trial.suggest_float('reg_lambda', 0.0, 1.0),
        'verbose': -1
    }

print("Training LightGBM (Poisson)...")
train_evaluate_regression_model_with_optuna(
    "LightGBM",
    lambda **params: LGBMRegressor(objective="poisson", **params),  # ✅ 加入 Poisson 目标
    lgbm_param_func,
    X, y, groups
)


[I 2025-05-11 19:45:40,585] A new study created in memory with name: no-name-8a9e7c6f-2903-4383-aa67-7c603137f69c


Training LightGBM (Poisson)...


Training LightGBM: 100%|██████████| 10/10 [00:09<00:00,  1.06it/s]
[I 2025-05-11 19:45:50,012] Trial 0 finished with value: 0.8651240755418668 and parameters: {'n_estimators': 299, 'max_depth': 7, 'num_leaves': 63, 'learning_rate': 0.1693366170580751, 'feature_fraction': 0.6206714815596162, 'bagging_fraction': 0.9065850180789002, 'bagging_freq': 2, 'reg_alpha': 0.8849241503220999, 'reg_lambda': 0.8479543128689718}. Best is trial 0 with value: 0.8651240755418668.
Training LightGBM: 100%|██████████| 10/10 [00:30<00:00,  3.02s/it]
[I 2025-05-11 19:46:20,266] Trial 1 finished with value: 0.8273383861146157 and parameters: {'n_estimators': 484, 'max_depth': 18, 'num_leaves': 155, 'learning_rate': 0.11192914088889441, 'feature_fraction': 0.7101373601583783, 'bagging_fraction': 0.7264201074634544, 'bagging_freq': 6, 'reg_alpha': 0.5667990297838863, 'reg_lambda': 0.7906055450742718}. Best is trial 1 with value: 0.8273383861146157.
Training LightGBM: 100%|██████████| 10/10 [00:05<00:00,  1.74it

Best parameters for LightGBM: {'n_estimators': 545, 'max_depth': 20, 'num_leaves': 234, 'learning_rate': 0.07843189293470809, 'feature_fraction': 0.8433583285342025, 'bagging_fraction': 0.8961655341447841, 'bagging_freq': 5, 'reg_alpha': 0.00021586877562570356, 'reg_lambda': 0.33340283254906883}
Best mean MAE: 0.8167


In [8]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import GroupKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error
import optuna
import numpy as np


class DNNWithSoftplus(nn.Module):
    def __init__(self, input_dim, hidden_sizes, activation):
        super().__init__()
        act_fn = {
            'relu': nn.ReLU(),
            'logistic': nn.Sigmoid(),
            'tanh': nn.Tanh()
        }[activation]
        layers = []
        prev_dim = input_dim
        for h in hidden_sizes:
            layers += [nn.Linear(prev_dim, h), act_fn]
            prev_dim = h
        layers += [nn.Linear(prev_dim, 1)]
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return F.softplus(self.net(x)).squeeze(-1)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
def train_dnn_with_optuna_pytorch(X, y, groups, device=device):
    def dnn_param_func(trial):
        return {
            'hidden_layer_sizes': trial.suggest_categorical(
                'hidden_layer_sizes', [(50,), (100,), (150,), (100, 50), (150, 100, 50)]
            ),
            'activation': trial.suggest_categorical('activation', ['relu', 'logistic', 'tanh']),
            'alpha': trial.suggest_float('alpha', 1e-5, 1e-2, log=True),
            'learning_rate': trial.suggest_float('learning_rate_init', 1e-4, 1e-2, log=True),
            'optimizer': trial.suggest_categorical('solver', ['adam', 'sgd'])
        }

    def objective(trial):
        params = dnn_param_func(trial)
        model = DNNWithSoftplus(
            input_dim=X.shape[1],
            hidden_sizes=params['hidden_layer_sizes'],
            activation=params['activation']
        ).to(device)

        optimizer = {
            'adam': torch.optim.Adam,
            'sgd': torch.optim.SGD
        }[params['optimizer']](model.parameters(), lr=params['learning_rate'], weight_decay=params['alpha'])

        loss_fn = nn.MSELoss()
        gkf = GroupKFold(n_splits=10)
        fold_maes = []

        for train_idx, val_idx in gkf.split(X, y, groups=groups):
            X_train, y_train = X[train_idx], y[train_idx]
            X_val, y_val = X[val_idx], y[val_idx]

            scaler = StandardScaler()
            X_train = scaler.fit_transform(X_train)
            X_val = scaler.transform(X_val)

            train_ds = TensorDataset(torch.tensor(X_train).float(), torch.tensor(y_train).float())
            train_loader = DataLoader(train_ds, batch_size=256, shuffle=True)

            model.train()
            for epoch in range(100):
                for xb, yb in train_loader:
                    xb, yb = xb.to(device), yb.to(device)
                    optimizer.zero_grad()
                    pred = model(xb)
                    loss = loss_fn(pred, yb)
                    loss.backward()
                    optimizer.step()

            model.eval()
            with torch.no_grad():
                val_preds = model(torch.tensor(X_val).float().to(device)).cpu().numpy()
                mae = mean_absolute_error(y_val, val_preds)
                fold_maes.append(mae)

        return np.mean(fold_maes)

    study = optuna.create_study(direction='minimize')
    study.optimize(objective, n_trials=30)
    print("\n✅ Best Parameters Found:")
    print(study.best_params)
    print(f"Mean MAE = {study.best_value:.4f}")
    return study.best_params


best_dnn_params = train_dnn_with_optuna_pytorch(X, y, groups)

[I 2025-05-15 21:59:56,763] A new study created in memory with name: no-name-5eae140c-8d3e-4d68-8ffa-b62d4f2418fa
[I 2025-05-15 22:02:00,054] Trial 0 finished with value: 0.8523309565614408 and parameters: {'hidden_layer_sizes': (100, 50), 'activation': 'tanh', 'alpha': 9.801360095855715e-05, 'learning_rate_init': 0.002580631513076238, 'solver': 'adam'}. Best is trial 0 with value: 0.8523309565614408.
[I 2025-05-15 22:04:03,735] Trial 1 finished with value: 0.4378747010659317 and parameters: {'hidden_layer_sizes': (100, 50), 'activation': 'relu', 'alpha': 1.023261795700467e-05, 'learning_rate_init': 0.0004398784187408231, 'solver': 'adam'}. Best is trial 1 with value: 0.4378747010659317.
[I 2025-05-15 22:06:00,954] Trial 2 finished with value: 0.5838586376666595 and parameters: {'hidden_layer_sizes': (150,), 'activation': 'relu', 'alpha': 2.693712793014769e-05, 'learning_rate_init': 0.003157517516207331, 'solver': 'adam'}. Best is trial 1 with value: 0.4378747010659317.
[I 2025-05-15 2


✅ Best Parameters Found:
{'hidden_layer_sizes': (100, 50), 'activation': 'relu', 'alpha': 1.023261795700467e-05, 'learning_rate_init': 0.0004398784187408231, 'solver': 'adam'}
Mean MAE = 0.4379
